# Project 3 - Dispatch Optimization Presentation

This notebook is organized for an oral walkthrough of the backend logic.

Presentation flow:
1. load the weekly and stochastic inputs;
2. validate the Bellman implementation on the 24-hour benchmark;
3. solve the 168-hour base case;
4. present each extension separately;
5. explain Jensen through the `E[D]` proxy;
6. replay the stochastic policy with Monte Carlo;
7. close with a comparison table and export-ready outputs.


## Reading Note

The code comments are intentionally minimal. The notebook is meant to be presented orally, so the structure is driven mainly by section headers and compact result tables.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd()
DATA_XLS = ROOT / "Project3data.xls"
VALIDATION_XLSM = ROOT / "Project3code_Student.xlsm"
OUTPUT_XLSX = ROOT / "dispatch_results.xlsx"

assert DATA_XLS.exists(), f"Missing file: {DATA_XLS}"
assert VALIDATION_XLSM.exists(), f"Missing file: {VALIDATION_XLSM}"

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)
DAYS = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
NEG_INF = -1e15

print(f"Working directory: {ROOT}")
print(f"Data file: {DATA_XLS.name}")
print(f"Validation file: {VALIDATION_XLSM.name}")


## 1. Helper Functions and Dynamic Programs

This cell contains the full backend logic used throughout the project: data loading, Bellman recursions, schedule extraction, reporting helpers, and the Monte Carlo replay function.


In [ ]:
# Data loading
def load_data(filepath=DATA_XLS):
    df = pd.read_excel(filepath, sheet_name="WeeklySchedule", header=None)
    margins = np.zeros(168)
    demands = np.zeros(168)
    startups = np.zeros(168)

    for day_idx in range(7):
        for hour_idx in range(24):
            row = hour_idx + 2
            t = day_idx * 24 + hour_idx
            margins[t] = float(df.iloc[row, 1 + day_idx])
            demands[t] = float(df.iloc[row, 10 + day_idx])
            startups[t] = float(df.iloc[row, 19 + day_idx])

    df_stoch = pd.read_excel(filepath, sheet_name="StochasticDemand", header=None)
    stoch_demand = []
    for hour_idx in range(24):
        row = hour_idx + 2
        D1 = float(df_stoch.iloc[row, 1])
        p1 = float(df_stoch.iloc[row, 2])
        D2 = float(df_stoch.iloc[row, 3])
        p2 = float(df_stoch.iloc[row, 4])
        stoch_demand.append((D1, p1, D2, p2))

    return margins, demands, startups, stoch_demand

# 24-hour benchmark loading
def load_24h_validation(filepath=VALIDATION_XLSM):
    df = pd.read_excel(filepath, sheet_name="24hours", header=None)
    margins_24 = np.zeros(24)
    demands_24 = np.full(24, 100.0)
    startups_24 = np.full(24, 400.0)
    ref_V_s0 = np.zeros(25)
    ref_V_s1 = np.zeros(25)
    ref_x_s0 = np.zeros(25)
    ref_x_s1 = np.zeros(25)

    for i in range(24):
        margins_24[i] = float(df.iloc[i + 1, 1])

    for i in range(25):
        ref_x_s0[i] = float(df.iloc[i + 1, 6]) if pd.notna(df.iloc[i + 1, 6]) else 0
        ref_x_s1[i] = float(df.iloc[i + 1, 7]) if pd.notna(df.iloc[i + 1, 7]) else 0
        ref_V_s0[i] = float(df.iloc[i + 1, 8]) if pd.notna(df.iloc[i + 1, 8]) else 0
        ref_V_s1[i] = float(df.iloc[i + 1, 9]) if pd.notna(df.iloc[i + 1, 9]) else np.nan

    return margins_24, demands_24, startups_24, ref_V_s0, ref_V_s1, ref_x_s0, ref_x_s1

# Base dynamic program
def solve_base_dp(margins, demands, startups):
    T = len(margins)
    V = np.zeros((T + 1, 2))
    X = np.zeros((T, 2), dtype=int)
    V[T, 0] = 0.0
    V[T, 1] = 0.0

    for t in range(T - 1, -1, -1):
        m = margins[t]
        D = demands[t]
        f = startups[t]

        val_stay_off = V[t + 1, 0]
        val_turn_on = m * D - f + V[t + 1, 1]
        if val_turn_on > val_stay_off:
            V[t, 0] = val_turn_on
            X[t, 0] = 1
        else:
            V[t, 0] = val_stay_off
            X[t, 0] = 0

        val_stay_on = m * D + V[t + 1, 1]
        val_turn_off = V[t + 1, 0]
        if val_stay_on >= val_turn_off:
            V[t, 1] = val_stay_on
            X[t, 1] = 0
        else:
            V[t, 1] = val_turn_off
            X[t, 1] = -1

    return V, X

# Base schedule reconstruction
def extract_schedule(X, s0=0):
    T = X.shape[0]
    states = np.zeros(T + 1, dtype=int)
    decisions = np.zeros(T, dtype=int)
    states[0] = s0

    for t in range(T):
        s = states[t]
        x = X[t, s]
        decisions[t] = x

        if s == 0 and x == 1:
            states[t + 1] = 1
        elif s == 1 and x == -1:
            states[t + 1] = 0
        else:
            states[t + 1] = s

    return states, decisions

# Extension 1: minimum runtime
def solve_ext1_min_runtime(margins, demands, startups, min_hours=10):
    T = len(margins)
    n_states = min_hours + 1
    V = np.full((T + 1, n_states), NEG_INF)
    X = np.zeros((T, n_states), dtype=int)
    V[T, 0] = 0.0

    for t in range(T - 1, -1, -1):
        m = margins[t]
        D = demands[t]
        f = startups[t]

        val_off = V[t + 1, 0]
        val_on = m * D - f + V[t + 1, 1]
        if val_on > val_off:
            V[t, 0] = val_on
            X[t, 0] = 1
        else:
            V[t, 0] = val_off
            X[t, 0] = 0

        for k in range(1, min_hours):
            V[t, k] = m * D + V[t + 1, k + 1]
            X[t, k] = 0

        val_stay = m * D + V[t + 1, min_hours]
        val_turn_off = V[t + 1, 0]
        if val_stay >= val_turn_off:
            V[t, min_hours] = val_stay
            X[t, min_hours] = 0
        else:
            V[t, min_hours] = val_turn_off
            X[t, min_hours] = -1

    return V, X

# Extension 1 reconstruction
def extract_schedule_ext1(X, min_hours=10):
    T = X.shape[0]
    states_detail = np.zeros(T + 1, dtype=int)
    states_binary = np.zeros(T + 1, dtype=int)
    decisions = np.zeros(T, dtype=int)

    for t in range(T):
        s = states_detail[t]
        x = X[t, s]
        decisions[t] = x

        if s == 0 and x == 1:
            states_detail[t + 1] = 1
        elif s == 0 and x == 0:
            states_detail[t + 1] = 0
        elif 1 <= s < min_hours:
            states_detail[t + 1] = s + 1
        elif s == min_hours and x == -1:
            states_detail[t + 1] = 0
        else:
            states_detail[t + 1] = min_hours

        states_binary[t + 1] = 1 if states_detail[t + 1] > 0 else 0

    return states_binary, decisions, states_detail

# Extension 2: weekly production cap
def solve_ext2_production_cap(margins, demands, startups, max_mwh=16500, step=50):
    T = len(margins)
    n_cum = max_mwh // step + 1
    V = np.full((T + 1, 2, n_cum), NEG_INF)
    X = np.zeros((T, 2, n_cum), dtype=int)
    V[T, 0, :] = 0.0
    V[T, 1, :] = NEG_INF

    for t in range(T - 1, -1, -1):
        m = margins[t]
        D = demands[t]
        f = startups[t]
        d_step = int(D / step)

        for c in range(n_cum):
            val_off = V[t + 1, 0, c]
            new_c = c + d_step
            val_on = m * D - f + V[t + 1, 1, new_c] if new_c < n_cum else NEG_INF
            if val_on > val_off:
                V[t, 0, c] = val_on
                X[t, 0, c] = 1
            else:
                V[t, 0, c] = val_off
                X[t, 0, c] = 0

            val_stay = m * D + V[t + 1, 1, new_c] if new_c < n_cum else NEG_INF
            val_turn_off = V[t + 1, 0, c]
            if val_stay >= val_turn_off:
                V[t, 1, c] = val_stay
                X[t, 1, c] = 0
            else:
                V[t, 1, c] = val_turn_off
                X[t, 1, c] = -1

    return V, X

# Extension 2 reconstruction
def extract_schedule_ext2(X, demands, step=50):
    T = X.shape[0]
    states = np.zeros(T + 1, dtype=int)
    cum = np.zeros(T + 1, dtype=int)
    decisions = np.zeros(T, dtype=int)

    for t in range(T):
        s = states[t]
        c = cum[t]
        x = X[t, s, c]
        decisions[t] = x
        d_step = int(demands[t] / step)

        if s == 0 and x == 1:
            states[t + 1] = 1
            cum[t + 1] = c + d_step
        elif s == 1 and x == 0:
            states[t + 1] = 1
            cum[t + 1] = c + d_step
        elif s == 1 and x == -1:
            states[t + 1] = 0
            cum[t + 1] = c
        else:
            states[t + 1] = 0
            cum[t + 1] = c

    return states, decisions, cum

# Extension 3: stochastic Sunday
def solve_ext3_stochastic(margins, demands, startups, stoch_demand, max_mwh=16500, step=50):
    T = len(margins)
    n_cum = max_mwh // step + 1
    V = np.full((T + 1, 2, n_cum), NEG_INF)
    X = np.zeros((T, 2, n_cum), dtype=int)
    V[T, 0, :] = 0.0
    V[T, 1, :] = NEG_INF

    for t in range(T - 1, -1, -1):
        m = margins[t]
        f = startups[t]
        day = t // 24
        hour_in_day = t % 24
        if day == 6:
            D1, p1, D2, p2 = stoch_demand[hour_in_day]
            scenarios = [(D1, p1), (D2, p2)]
        else:
            scenarios = [(demands[t], 1.0)]

        for c in range(n_cum):
            ev_off = V[t + 1, 0, c]

            ev_on = 0.0
            feasible_on = True
            for D_val, prob in scenarios:
                new_c = c + int(D_val / step)
                if new_c < n_cum:
                    ev_on += prob * (m * D_val - f + V[t + 1, 1, new_c])
                else:
                    feasible_on = False
                    break
            if not feasible_on:
                ev_on = NEG_INF

            if ev_on > ev_off:
                V[t, 0, c] = ev_on
                X[t, 0, c] = 1
            else:
                V[t, 0, c] = ev_off
                X[t, 0, c] = 0

            ev_stay = 0.0
            feasible_stay = True
            for D_val, prob in scenarios:
                new_c = c + int(D_val / step)
                if new_c < n_cum:
                    ev_stay += prob * (m * D_val + V[t + 1, 1, new_c])
                else:
                    feasible_stay = False
                    break
            if not feasible_stay:
                ev_stay = NEG_INF

            ev_turn_off = V[t + 1, 0, c]
            if ev_stay >= ev_turn_off:
                V[t, 1, c] = ev_stay
                X[t, 1, c] = 0
            else:
                V[t, 1, c] = ev_turn_off
                X[t, 1, c] = -1

    return V, X

# Monte Carlo replay of the Extension 3 policy
def simulate_ext3_monte_carlo(X_policy, margins, demands, startups, stoch_demand, iterations=5000, seed=42, step=50):
    """
    Rejoue la politique optimale de l'extension 3 sur de nombreuses
    réalisations aléatoires du dimanche.
    """
    rng = np.random.default_rng(seed)
    T = len(margins)
    profits = np.zeros(iterations)
    final_production = np.zeros(iterations)

    for path in range(iterations):
        state = 0
        cum = 0
        profit = 0.0

        for t in range(T):
            day = t // 24
            hour_in_day = t % 24
            realized_demand = demands[t]

            if day == 6:
                D1, p1, D2, p2 = stoch_demand[hour_in_day]
                realized_demand = D1 if rng.random() < p1 else D2

            decision = X_policy[t, state, cum]
            d_step = int(realized_demand / step)

            if state == 0 and decision == 1:
                profit += margins[t] * realized_demand - startups[t]
                state = 1
                cum += d_step
            elif state == 1 and decision == 0:
                profit += margins[t] * realized_demand
                cum += d_step
            elif state == 1 and decision == -1:
                state = 0

        profits[path] = profit
        final_production[path] = cum * step

    counts, bin_edges = np.histogram(profits, bins=24)
    summary = pd.DataFrame([
        {"metric": "Iterations", "value": float(iterations)},
        {"metric": "Mean profit", "value": float(profits.mean())},
        {"metric": "Std. dev.", "value": float(profits.std())},
        {"metric": "P05", "value": float(np.percentile(profits, 5))},
        {"metric": "Median", "value": float(np.percentile(profits, 50))},
        {"metric": "P95", "value": float(np.percentile(profits, 95))},
        {"metric": "Min", "value": float(profits.min())},
        {"metric": "Max", "value": float(profits.max())},
        {"metric": "Average production (MWh)", "value": float(final_production.mean())},
    ])
    histogram = pd.DataFrame({
        "bin_left": bin_edges[:-1],
        "bin_right": bin_edges[1:],
        "count": counts,
    })

    return {
        "profits": profits,
        "final_production": final_production,
        "summary": summary,
        "histogram": histogram,
    }

def reshape_week(values, index_name="Hour"):
    arr = np.asarray(values).reshape(7, 24).T
    return pd.DataFrame(arr, index=pd.Index(range(1, 25), name=index_name), columns=DAYS)

def build_terminal_row(V):
    return pd.DataFrame({"state": [0, 1], "V_terminal": [V[-1, 0], V[-1, 1]]})

def schedule_status(states):
    status = []
    for t in range(len(states) - 1):
        if states[t] == 0 and states[t + 1] == 1:
            status.append("ON*")
        elif states[t + 1] == 1:
            status.append("ON")
        else:
            status.append(".")
    return np.array(status)

def hourly_profit(states, margins, demands, startups):
    profit = np.zeros(len(states) - 1)
    for t in range(len(states) - 1):
        if states[t] == 0 and states[t + 1] == 1:
            profit[t] = margins[t] * demands[t] - startups[t]
        elif states[t + 1] == 1:
            profit[t] = margins[t] * demands[t]
    return profit

def build_hourly_schedule_table(states, margins, demands, startups):
    status = schedule_status(states)
    profit = hourly_profit(states, margins, demands, startups)
    rows = []
    for t in range(len(status)):
        rows.append(
            {
                "t": t + 1,
                "day": DAYS[t // 24],
                "hour": t % 24 + 1,
                "status": status[t],
                "margin_eur_per_mwh": margins[t],
                "demand_mw": demands[t],
                "startup_cost_eur": startups[t],
                "hourly_profit_eur": profit[t],
            }
        )
    return pd.DataFrame(rows)

def summarize_case(states, margins, demands, startups, label):
    profit = hourly_profit(states, margins, demands, startups)
    status = schedule_status(states)
    total_mwh = float(sum(demands[t] for t in range(len(status)) if states[t + 1] == 1))
    return {
        "case": label,
        "profit_eur": float(profit.sum()),
        "production_mwh": total_mwh,
        "startups": int((status == "ON*").sum()),
    }


## 2. Weekly Dataset and Input Structure

We first load the deterministic weekly inputs and the stochastic Sunday table. The stochastic table is an exogenous input: for each Sunday hour, it provides two demand realizations and their probabilities.


In [ ]:
margins, demands, startups, stoch_demand = load_data()
margins_24, demands_24, startups_24, ref_V_s0, ref_V_s1, ref_x_s0, ref_x_s1 = load_24h_validation()

weekly_overview = pd.DataFrame(
    {
        "series": ["Margins", "Demand", "Startup cost"],
        "min": [margins.min(), demands.min(), startups.min()],
        "max": [margins.max(), demands.max(), startups.max()],
        "mean": [margins.mean(), demands.mean(), startups.mean()],
    }
)

stochastic_sunday_table = pd.DataFrame(
    stoch_demand,
    columns=["D1", "p1", "D2", "p2"],
    index=np.arange(1, 25),
)
stochastic_sunday_table.index.name = "Sunday hour"

display(Markdown("**Weekly deterministic overview**"))
display(weekly_overview)
display(Markdown("**Sunday stochastic input**"))
display(stochastic_sunday_table)


## 3. 24-Hour Benchmark Validation

Before solving the full week, we verify that the Bellman implementation reproduces the benchmark case from the course material. This is the key consistency check for the whole notebook.


In [ ]:
V_24, X_24 = solve_base_dp(margins_24, demands_24, startups_24)

validation_summary = pd.DataFrame(
    {
        "metric": ["Calculated V1(0)", "Reference V1(0)", "Absolute gap"],
        "value": [V_24[0, 0], ref_V_s0[0], abs(V_24[0, 0] - ref_V_s0[0])],
    }
)

compare_24h = pd.DataFrame(
    {
        "hour": np.arange(1, 25),
        "margin": margins_24,
        "x_calc_s0": X_24[:, 0],
        "x_ref_s0": ref_x_s0[:24].astype(int),
        "match_s0": X_24[:, 0] == ref_x_s0[:24].astype(int),
        "V_calc_s0": V_24[:24, 0],
        "V_ref_s0": ref_V_s0[:24],
    }
)

display(validation_summary)
display(compare_24h)


## 4. Base Case: 168-Hour Dispatch Problem

The base case is the standard binary dispatch problem. The state is only `OFF/ON`, and the plant must finish the week in the `OFF` state.


In [ ]:
V_base, X_base = solve_base_dp(margins, demands, startups)
states_base, decisions_base = extract_schedule(X_base, s0=0)

base_summary = pd.DataFrame([
    summarize_case(states_base, margins, demands, startups, "Base 168h")
])
base_planning = reshape_week(schedule_status(states_base))
base_x_s0 = reshape_week(X_base[:, 0])
base_x_s1 = reshape_week(X_base[:, 1])
base_V_s0 = reshape_week(np.round(V_base[:168, 0], 2))
base_V_s1 = reshape_week(np.round(V_base[:168, 1], 2))
base_terminal = build_terminal_row(V_base)
base_hourly_table = build_hourly_schedule_table(states_base, margins, demands, startups)

display(base_summary)
display(Markdown("**Base weekly planning**"))
display(base_planning)
display(Markdown("**Base decisions x_t(s=0)**"))
display(base_x_s0)
display(Markdown("**Base decisions x_t(s=1)**"))
display(base_x_s1)
display(Markdown("**Base value function V_t(s=0)**"))
display(base_V_s0)
display(Markdown("**Base value function V_t(s=1)**"))
display(base_V_s1)
display(Markdown("**Base terminal condition**"))
display(base_terminal)


## 5. Extension 1: Minimum Runtime Constraint

Extension 1 changes the state space. Once the plant starts, it must remain on for a minimum number of hours before shutdown becomes admissible again.


In [ ]:
MIN_RUNTIME = 10

V_ext1, X_ext1 = solve_ext1_min_runtime(margins, demands, startups, min_hours=MIN_RUNTIME)
states_ext1, decisions_ext1, states_detail_ext1 = extract_schedule_ext1(X_ext1, min_hours=MIN_RUNTIME)

ext1_summary = pd.DataFrame([
    summarize_case(states_ext1, margins, demands, startups, f"Ext 1 - min {MIN_RUNTIME}h")
])
ext1_planning = reshape_week(schedule_status(states_ext1))
ext1_state_table = pd.DataFrame(
    {
        "t": np.arange(1, 169),
        "detailed_state": states_detail_ext1[1:],
        "binary_state": states_ext1[1:],
        "decision": decisions_ext1,
    }
)

display(ext1_summary)
display(Markdown("**Extension 1 planning**"))
display(ext1_planning)
display(Markdown("**Extension 1 state tracking**"))
display(ext1_state_table.head(24))


## 6. Extension 2: Weekly Production Cap

Extension 2 adds cumulative production to the Bellman state. The cap is weekly, not daily, so the solver must track how much production budget has already been consumed.


In [ ]:
MAX_MWH = 16500
STEP_MWH = 50

V_ext2, X_ext2 = solve_ext2_production_cap(margins, demands, startups, max_mwh=MAX_MWH, step=STEP_MWH)
states_ext2, decisions_ext2, cum_ext2 = extract_schedule_ext2(X_ext2, demands, step=STEP_MWH)

ext2_summary = pd.DataFrame([
    summarize_case(states_ext2, margins, demands, startups, f"Ext 2 - cap {MAX_MWH}")
])
ext2_planning = reshape_week(schedule_status(states_ext2))
ext2_cum_table = pd.DataFrame(
    {
        "t": np.arange(1, 169),
        "cum_step_50": cum_ext2[1:],
        "cum_mwh": cum_ext2[1:] * STEP_MWH,
    }
)

display(ext2_summary)
display(Markdown("**Extension 2 planning**"))
display(ext2_planning)
display(Markdown("**Extension 2 cumulative production**"))
display(ext2_cum_table)


## 7. Extension 3: Stochastic Sunday

Extension 3 keeps the weekly cap from Extension 2 and makes Sunday demand stochastic. For each Sunday hour, Bellman uses two possible demand realizations, weighted by their probabilities.


In [ ]:
# Exact stochastic Bellman
V_ext3, X_ext3 = solve_ext3_stochastic(
    margins,
    demands,
    startups,
    stoch_demand,
    max_mwh=MAX_MWH,
    step=STEP_MWH,
)

# Deterministic proxy used for the Jensen comparison
expected_demands = demands.copy()
for h in range(24):
    D1, p1, D2, p2 = stoch_demand[h]
    expected_demands[144 + h] = D1 * p1 + D2 * p2

V_ext3_det, _ = solve_ext2_production_cap(
    margins,
    expected_demands,
    startups,
    max_mwh=MAX_MWH,
    step=STEP_MWH,
)

ext3_comparison = pd.DataFrame(
    [
        {"case": "Ext 3 exact stochastic Bellman", "profit_eur": float(V_ext3[0, 0, 0])},
        {"case": "Ext 3 with expected demand E[D]", "profit_eur": float(V_ext3_det[0, 0, 0])},
        {"case": "Jensen gap (exact - E[D] proxy)", "profit_eur": float(V_ext3[0, 0, 0] - V_ext3_det[0, 0, 0])},
    ]
)

expected_sunday_table = pd.DataFrame(
    {
        "D1": [row[0] for row in stoch_demand],
        "p1": [row[1] for row in stoch_demand],
        "D2": [row[2] for row in stoch_demand],
        "p2": [row[3] for row in stoch_demand],
        "E[D]": [row[0] * row[1] + row[2] * row[3] for row in stoch_demand],
    },
    index=np.arange(1, 25),
)
expected_sunday_table.index.name = "Sunday hour"

display(ext3_comparison)
display(Markdown("**Expected-demand proxy used in the Jensen comparison**"))
display(expected_sunday_table)


## 8. Jensen Interpretation

The Jensen comparison is:
- `E[V(D)]`: the exact stochastic Bellman;
- `V(E[D])`: the deterministic Bellman solved on the expected Sunday demand.

Because the production cap makes the problem nonlinear, the two values are not equal. In practice, the expected-demand proxy is too optimistic.


In [ ]:
jensen_summary = pd.DataFrame(
    {
        "quantity": [
            "Exact stochastic Bellman E[V(D)]",
            "Deterministic proxy V(E[D])",
            "Jensen gap E[V(D)] - V(E[D])",
        ],
        "value_eur": [
            float(V_ext3[0, 0, 0]),
            float(V_ext3_det[0, 0, 0]),
            float(V_ext3[0, 0, 0] - V_ext3_det[0, 0, 0]),
        ],
    }
)

display(jensen_summary)


## 9. Monte Carlo Replay of the Stochastic Policy

Monte Carlo is added on top of Bellman. It does not re-optimize the problem. It simply replays the optimal stochastic policy on many random Sunday realizations in order to visualize dispersion, tail risk, and average realized profit.


In [ ]:
MC_ITERATIONS = 5000
MC_SEED = 42

mc_ext3 = simulate_ext3_monte_carlo(
    X_ext3,
    margins,
    demands,
    startups,
    stoch_demand,
    iterations=MC_ITERATIONS,
    seed=MC_SEED,
    step=STEP_MWH,
)

display(Markdown("**Monte Carlo summary**"))
display(mc_ext3["summary"])
display(Markdown("**Monte Carlo histogram table**"))
display(mc_ext3["histogram"])


## 10. Final Comparison Across All Cases

This table is the cleanest closing slide for the oral presentation: it shows how each modeling layer changes the optimal profit compared with the base case.


In [ ]:
summary_rows = [
    summarize_case(states_base, margins, demands, startups, "Base"),
    summarize_case(states_ext1, margins, demands, startups, "Ext 1 - minimum runtime"),
    summarize_case(states_ext2, margins, demands, startups, "Ext 2 - production cap"),
    {
        "case": "Ext 3 - stochastic Sunday",
        "profit_eur": float(V_ext3[0, 0, 0]),
        "production_mwh": np.nan,
        "startups": np.nan,
    },
    {
        "case": "Ext 3 with E[D] proxy",
        "profit_eur": float(V_ext3_det[0, 0, 0]),
        "production_mwh": np.nan,
        "startups": np.nan,
    },
]

presentation_summary = pd.DataFrame(summary_rows)
presentation_summary["delta_vs_base"] = presentation_summary["profit_eur"] - presentation_summary.loc[0, "profit_eur"]
display(presentation_summary)


## 11. Export-Ready Deliverables

The notebook still produces the same export package as the working notebook. This keeps the presentation version aligned with the deliverable workflow.


In [ ]:
with pd.ExcelWriter(OUTPUT_XLSX, engine="openpyxl") as writer:
    validation_summary.to_excel(writer, sheet_name="Validation24h", index=False)
    compare_24h.to_excel(writer, sheet_name="Validation24h_Compare", index=False)
    base_summary.to_excel(writer, sheet_name="BaseSummary", index=False)
    base_x_s0.to_excel(writer, sheet_name="Base_X_s0")
    base_x_s1.to_excel(writer, sheet_name="Base_X_s1")
    base_V_s0.to_excel(writer, sheet_name="Base_V_s0")
    base_V_s1.to_excel(writer, sheet_name="Base_V_s1")
    base_terminal.to_excel(writer, sheet_name="Base_Terminal", index=False)
    base_planning.to_excel(writer, sheet_name="Base_Planning")
    base_hourly_table.to_excel(writer, sheet_name="Base_Hourly", index=False)
    ext1_planning.to_excel(writer, sheet_name="Ext1_Planning")
    ext2_planning.to_excel(writer, sheet_name="Ext2_Planning")
    ext2_cum_table.to_excel(writer, sheet_name="Ext2_Cum", index=False)
    presentation_summary.to_excel(writer, sheet_name="Extensions_Summary", index=False)
    mc_ext3["summary"].to_excel(writer, sheet_name="Ext3_MC_Summary", index=False)
    mc_ext3["histogram"].to_excel(writer, sheet_name="Ext3_MC_Hist", index=False)

print(f"Results exported to: {OUTPUT_XLSX}")
